# Artificial and Computational Intelligence
## Assignment 1: Autonomous battery-operated micro aquatic boat
**Group ID: G082**

### Team Contribution

| Name | BITS ID | Percentage Contribution |
| :--- | :--- | :--- |
| PRABHABATI MOHARANA | 2025Ag05430 | 20% |
| SAUMYAJEET MOHANTY | 2025ag05436 | 20% |
| MALAYA SETHI | 2025ag05448 | 20% |
| SRIMAN SASWAT SUVANKAR | 2025ag05568 | 20% |
| SUBHAM SARAT SWAIN | 2025ag05650 | 20% |

### Problem Statement

Assume that you are given the responsibility to supply food, water, medicines and other amenities to the flooded areas in the city of Chennai. It is your responsibility to supply the needs to all the areas affected by flood where people are suffering. You are provided with a map of the city with flooded land-marks marked (vertices). You are provided with an autonomous battery-operated micro aquatic boat that detects the flooded areas and responds to people who wave to it for help. The autonomous battery-operated micro aquatic boat works on a battery and hence it has to take a path such that all the roads (edges in the graph) are covered, but no road is repeated more than once. After getting the details, the boat agent reports to you, the taken path and the locations where people are requesting for help.[cite: 5]

The problem here is to find the shortest route that travels through all the lanes in the area. The places can be visited more than once. The shortest path includes all the roads travelled only once. Help your autonomous battery-operated micro aquatic boat in finding such a path given a starting point and the map.[cite: 5]

Here the area map is represented as a graph. The algorithm takes the starting point and the graph as the input and produces the shortest path covering all the edges only once.[cite: 5]

### Problem Formulation

* **Initial State (IS):** The autonomous boat starts at a designated landmark (e.g., Nungambakkam) with an empty record of travelled roads and full battery capacity.
* **Possible Actions (Operators):** Moving the boat from its current vertex (landmark) to any adjacent connected vertex via a flooded road.
* **Successor Function / Transition Model:** `RESULT(current_vertex, move_action)` updates the boat's location to the next vertex and adds the specific connecting road to the boat's internal tracker. The transition model permits visiting a vertex more than once but strictly prevents traversing an already recorded road.
* **Goal Test:** Checks if the total count of uniquely traversed roads matches the exact total number of edges in the provided graph map (9 edges).
* **Path Cost:** The sum of the weights (battery usage/distance) of every road traversed to reach the current state.

In [71]:
# ==========================================
# Imports and Graph Setup
# Purpose: Imports required libraries and defines the bidirectional graph
# representing the flooded Chennai map. Calculates the total number of edges.
# ==========================================

import heapq
import random
import time
import tracemalloc

# Bidirectional Graph Definition based on the provided map
graph = {
    "Nungambakkam": {"Guindy": 12, "Tambaram": 15},
    "Guindy": {"Nungambakkam": 12, "Velachery": 14, "Perumbakkam": 7, "Tambaram": 10},
    "Velachery": {"Guindy": 14, "Perumbakkam": 6},
    "Perumbakkam": {"Guindy": 7, "Velachery": 6, "Tambaram": 11, "Purasawalkam": 8},
    "Tambaram": {"Nungambakkam": 15, "Guindy": 10, "Perumbakkam": 11, "Purasawalkam": 9},
    "Purasawalkam": {"Tambaram": 9, "Perumbakkam": 8}
}

# The total number of unique roads (edges) that the boat must cover
total_edges = 9

In [72]:
# ==========================================
# Greedy Best First Search Algorithm
# Purpose: Implements an informed search to find a path that covers all edges.
# Heuristic used: Number of unvisited edges remaining (lower is better).
# ==========================================

def greedy_best_first_edge_cover(start_node):
    # Priority Queue stores: (heuristic, current_node, path_sequence, visited_edges_set)
    pq = []
    heapq.heappush(pq, (total_edges, start_node, [start_node], set()))

    while pq:
        h, curr, path, visited = heapq.heappop(pq)

        # Goal Test: All edges covered exactly once
        if len(visited) == total_edges:
            return path

        for neighbor, weight in graph[curr].items():
            edge = tuple(sorted((curr, neighbor)))

            # Ensure no road is repeated more than once
            if edge not in visited:
                new_visited = set(visited)
                new_visited.add(edge)

                # Heuristic: Remaining unvisited edges
                new_h = total_edges - len(new_visited)

                heapq.heappush(pq, (new_h, neighbor, path + [neighbor], new_visited))

    return None

In [73]:
# ==========================================
# Genetic Algorithm
# Purpose: Implements a local search via evolutionary population generation.
# Fitness is based on maximizing unique edges visited without repeating them
# or taking invalid graph paths.
# ==========================================

def generate_random_walk(start, length):
    """Generates a random valid path based on graph connections."""
    path = [start]
    curr = start
    for _ in range(length - 1):
        curr = random.choice(list(graph[curr].keys()))
        path.append(curr)
    return path

def fitness(path):
    """Evaluates how well a path covers unique edges without repeating."""
    visited = set()
    penalty = 0
    for i in range(len(path) - 1):
        u, v = path[i], path[i+1]

        # Check if the move is valid in the graph
        if v not in graph.get(u, {}):
            penalty += 50  # Heavy penalty for invalid moves (non-existent roads)
        else:
            edge = tuple(sorted((u, v)))
            if edge in visited:
                penalty += 10 # Penalty for repeating a road
            visited.add(edge)

    # Reward unique edges, subtract penalties
    return len(visited) - penalty

def genetic_algorithm_edge_cover(start, generations=100, pop_size=100):
    # Chromosome length is total_edges + 1 (10 vertices needed to cover 9 edges)
    chrom_length = total_edges + 1

    # Initialize population with valid random walks
    population = [generate_random_walk(start, chrom_length) for _ in range(pop_size)]

    for _ in range(generations):
        # Sort population by highest fitness
        population.sort(key=fitness, reverse=True)

        # Goal Test during generations
        if fitness(population[0]) == total_edges:
            return population[0]

        # Keep the top half (elitism)
        next_gen = population[:pop_size // 2]

        # Mutation-driven regeneration to respect graph connectivity
        while len(next_gen) < pop_size:
            parent = random.choice(population[:20])
            mutate_point = random.randint(1, chrom_length - 2)

            # Splice a new random walk from the mutation point to ensure path validity
            new_tail = generate_random_walk(parent[mutate_point], chrom_length - mutate_point)
            child = parent[:mutate_point] + new_tail
            next_gen.append(child)

        population = next_gen

    return population[0]

In [74]:
# ==========================================
# Execution and Complexity Tracking
# Purpose: Runs both algorithms from the starting point and calculates
# the space and time complexity using tracemalloc and time modules.
# ==========================================

if __name__ == "__main__":
    start_point = "Nungambakkam"

    print("--- Greedy Best First Search ---")
    tracemalloc.start()
    start_time = time.time()

    gbfs_path = greedy_best_first_edge_cover(start_point)

    gbfs_time = time.time() - start_time
    _, gbfs_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print(f"Path taken by agent: {gbfs_path}")
    print(f"Time Complexity: {gbfs_time:.6f} seconds")
    print(f"Space Complexity: {gbfs_peak} bytes\n")

    print("--- Genetic Algorithm ---")
    tracemalloc.start()
    start_time = time.time()

    ga_path = genetic_algorithm_edge_cover(start_point)

    ga_time = time.time() - start_time
    _, ga_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    print(f"Path taken by agent: {ga_path}")
    print(f"Time Complexity: {ga_time:.6f} seconds")
    print(f"Space Complexity: {ga_peak} bytes")

--- Greedy Best First Search ---
Path taken by agent: ['Nungambakkam', 'Guindy', 'Perumbakkam', 'Purasawalkam', 'Tambaram', 'Guindy', 'Velachery', 'Perumbakkam', 'Tambaram', 'Nungambakkam']
Time Complexity: 0.000115 seconds
Space Complexity: 4616 bytes

--- Genetic Algorithm ---
Path taken by agent: ['Nungambakkam', 'Guindy', 'Perumbakkam', 'Tambaram', 'Purasawalkam', 'Perumbakkam', 'Velachery', 'Guindy', 'Tambaram', 'Nungambakkam']
Time Complexity: 0.002625 seconds
Space Complexity: 27088 bytes


In [75]:
# ==========================================
# Dynamic Observation & Justification
# Purpose: Renders final observations as Markdown using the calculated time and space variables
# ==========================================
from IPython.display import display, Markdown

observation_text = f"""
### Observation and Justification

* **Successful Edge Coverage (Eulerian Circuit):** Both algorithms successfully found a path that visits 10 vertices to cover all 9 edges exactly once.
* **Multiple Optimal Paths:** The algorithms often find different valid routes (e.g., branching at Guindy). This proves that the graph contains multiple optimal solutions, and both approaches are robust enough to find one.
* **Time Complexity Analysis:** GBFS ({gbfs_time:.6f} seconds) is significantly faster than GA ({ga_time:.6f} seconds). GA takes much longer to execute because evaluating fitness for an entire population across multiple generations is computationally heavy, whereas GBFS makes a single, fast greedy choice at each node.
* **Space Complexity Analysis:** GBFS ({gbfs_peak} bytes) is highly memory-efficient compared to GA ({ga_peak} bytes). GA consumes more memory because it must store and manipulate a large matrix of chromosomes (paths) simultaneously, while GBFS only maintains a lightweight priority queue.
"""

display(Markdown(observation_text))


### Observation and Justification

* **Successful Edge Coverage (Eulerian Circuit):** Both algorithms successfully found a path that visits 10 vertices to cover all 9 edges exactly once. 
* **Multiple Optimal Paths:** The algorithms often find different valid routes (e.g., branching at Guindy). This proves that the graph contains multiple optimal solutions, and both approaches are robust enough to find one.
* **Time Complexity Analysis:** GBFS (0.000115 seconds) is significantly faster than GA (0.002625 seconds). GA takes much longer to execute because evaluating fitness for an entire population across multiple generations is computationally heavy, whereas GBFS makes a single, fast greedy choice at each node.
* **Space Complexity Analysis:** GBFS (4616 bytes) is highly memory-efficient compared to GA (27088 bytes). GA consumes more memory because it must store and manipulate a large matrix of chromosomes (paths) simultaneously, while GBFS only maintains a lightweight priority queue. 
